## Experiment: Informed Search Techniques (Greedy Best-First Search and A* Algorithm)

This experiment focuses on implementing and understanding informed search algorithms, specifically A* search, and the crucial role of heuristic functions in guiding the search and improving efficiency. We'll start with a simple pathfinding example and then move to the classic 8-puzzle problem.

### Simple Code: A* Pathfinding on a Grid (Conceptual Understanding)

This simple example demonstrates the A* search algorithm for finding the shortest path on a small grid. It highlights how a heuristic function (Manhattan distance) estimates the cost to the goal, guiding the search more efficiently than uninformed methods. Each step costs 1 unit.

In [1]:
import heapq # Used for implementing the priority queue (min-heap) for A* algorithm

# Define the grid map
# 0: open path, 1: obstacle
grid = [
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0) # Starting coordinates (row, column)
goal = (4, 4)  # Goal coordinates

# Heuristic function (h(n)): Manhattan Distance
# This estimates the distance from the current node 'a' to the 'goal' node.
# It's admissible (never overestimates the actual cost) and consistent for grid movement.
def manhattan_distance(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# A* search algorithm implementation
def a_star_search(grid, start, goal):
    rows, cols = len(grid), len(grid[0])
    # priority_queue stores (f_score, g_score, current_node, path)
    # f_score = g_score + h_score, where g_score is actual cost and h_score is heuristic
    priority_queue = [(0 + manhattan_distance(start, goal), 0, start, [start])]
    # g_scores stores the cheapest cost from start to a node found so far
    g_scores = {start: 0}
    # visited set to keep track of nodes already processed (for efficiency)
    visited = set()

    # Loop until the priority queue is empty or the goal is found
    while priority_queue:
        # Pop the node with the lowest f_score
        f_score, g_score, current_node, path = heapq.heappop(priority_queue)

        # If we reached the goal, return the path
        if current_node == goal:
            return path

        # If the node has already been visited with a better or equal path, skip
        if current_node in visited:
            continue
        visited.add(current_node)

        # Define possible movements (up, down, left, right)
        # dr: delta row, dc: delta column
        moves = [(0, 1), (0, -1), (1, 0), (-1, 0)]

        # Explore neighbors
        for dr, dc in moves:
            neighbor_r, neighbor_c = current_node[0] + dr, current_node[1] + dc
            neighbor = (neighbor_r, neighbor_c)

            # Check if the neighbor is within grid boundaries and is not an obstacle
            if 0 <= neighbor_r < rows and 0 <= neighbor_c < cols and grid[neighbor_r][neighbor_c] == 0:
                # Calculate the cost to reach this neighbor from the start
                new_g_score = g_score + 1 # Each step costs 1

                # If this new path to the neighbor is shorter than any previously found path,
                # or if the neighbor hasn't been visited yet
                if neighbor not in g_scores or new_g_score < g_scores[neighbor]:
                    g_scores[neighbor] = new_g_score # Update the g_score for the neighbor
                    # Calculate the f_score for the neighbor
                    f_score_neighbor = new_g_score + manhattan_distance(neighbor, goal)
                    # Add the neighbor to the priority queue with its f_score, g_score, and updated path
                    heapq.heappush(priority_queue, (f_score_neighbor, new_g_score, neighbor, path + [neighbor]))

    return None # No path found

# Find the path using A*
path = a_star_search(grid, start, goal)

# Display the grid and the found path
if path:
    print("Path found:")
    # Create a copy of the grid to mark the path
    display_grid = [row[:] for row in grid]
    for r, c in path:
        # Mark path with 'P'
        display_grid[r][c] = 'P'
    display_grid[start[0]][start[1]] = 'S' # Mark start with 'S'
    display_grid[goal[0]][goal[1]] = 'G'   # Mark goal with 'G'

    # Print the grid in a readable format
    for row in display_grid:
        print(" ".join(map(str, row)))
    print(f"Total steps: {len(path) - 1}")
else:
    print("No path found.")


Path found:
S P P 1 0
0 1 P P P
0 0 0 1 P
0 1 0 0 P
0 0 0 0 G
Total steps: 8


### Complex Application: Solving the 8-Puzzle with A* Algorithm (Real-World Application)

Informed search algorithms like A* are powerful tools for solving complex problems. The 8-puzzle is a classic example in AI for demonstrating search strategies. The goal is to rearrange a set of 8 numbered tiles in a 3x3 grid into a specific target configuration by sliding the blank space (represented by 0).

Here, A* uses a heuristic function (Manhattan distance, sum of distances of each tile from its goal position) to efficiently find the optimal solution path (sequence of moves) from an initial state to the goal state.

In [3]:
import heapq # Used for the priority queue in A* search

# Define the goal state for the 8-puzzle
# Tiles are numbered 1-8, with 0 representing the blank space.
# The goal is (1, 2, 3, 4, 5, 6, 7, 8, 0)
GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# Function to print the puzzle state in a readable 3x3 format
def print_puzzle(state):
    for i in range(3):
        print(state[i*3 : (i+1)*3]) # Slice the tuple to get each row
    print()

# Heuristic function (h(n)): Manhattan Distance
# This calculates the sum of the Manhattan distances of each tile from its correct position in the goal state.
# It's an admissible heuristic for the 8-puzzle.
def manhattan_distance_heuristic(state):
    distance = 0
    # Iterate through each tile in the current state
    for i in range(9):
        tile = state[i]
        if tile == 0: # Ignore the blank tile
            continue
        # Find the current row and column of the tile
        current_row, current_col = i // 3, i % 3
        # Find the goal row and column of the tile
        # We need to find the index of 'tile' in GOAL_STATE to get its goal position
        goal_index = GOAL_STATE.index(tile)
        goal_row, goal_col = goal_index // 3, goal_index % 3
        # Add the Manhattan distance for this tile to the total distance
        distance += abs(current_row - goal_row) + abs(current_col - goal_col)
    return distance

# A* search algorithm for the 8-puzzle
def a_star_8_puzzle(initial_state):
    # Convert initial state to a tuple to make it hashable for set/dict keys
    initial_state = tuple(initial_state)

    # If the initial state is already the goal, return immediately
    if initial_state == GOAL_STATE:
        return [initial_state]

    # priority_queue stores (f_score, g_score, current_state, path_to_current_state)
    # f_score = g_score + h_score
    priority_queue = [(manhattan_distance_heuristic(initial_state), 0, initial_state, [initial_state])]

    # g_scores: stores the cheapest cost from initial state to a node found so far
    g_scores = {initial_state: 0}

    # visited: set to keep track of states already processed (to avoid redundant work)
    visited = set()

    # Loop until the priority queue is empty or the goal is found
    while priority_queue:
        # Pop the state with the lowest f_score
        f_score, g_score, current_state, path = heapq.heappop(priority_queue)

        # If we reached the goal state, return the path
        if current_state == GOAL_STATE:
            return path

        # If the state has already been visited with a better or equal path, skip
        if current_state in visited:
            continue
        visited.add(current_state)

        # Find the position of the blank tile (0)
        blank_index = current_state.index(0)
        blank_row, blank_col = blank_index // 3, blank_index % 3

        # Define possible moves (up, down, left, right for the blank tile)
        # Each move consists of (delta_row, delta_col)
        moves = [(0, 1), (0, -1), (1, 0), (-1, 0)]

        # Explore neighbors (possible next states)
        for dr, dc in moves:
            new_blank_row, new_blank_col = blank_row + dr, blank_col + dc

            # Check if the new blank position is within grid boundaries
            if 0 <= new_blank_row < 3 and 0 <= new_blank_col < 3:
                # Calculate the index of the tile to swap with the blank
                swap_index = new_blank_row * 3 + new_blank_col

                # Create a new state by swapping the blank with the tile at swap_index
                new_state_list = list(current_state)
                new_state_list[blank_index], new_state_list[swap_index] = new_state_list[swap_index], new_state_list[blank_index]
                new_state = tuple(new_state_list)

                # Calculate the cost to reach this new state (1 for each move)
                new_g_score = g_score + 1

                # If this new path to the neighbor state is shorter than any previously found path,
                # or if the state hasn't been visited yet
                if new_state not in g_scores or new_g_score < g_scores[new_state]:
                    g_scores[new_state] = new_g_score # Update the g_score for the new state
                    # Calculate the f_score for the new state
                    f_score_new_state = new_g_score + manhattan_distance_heuristic(new_state)
                    # Add the new state to the priority queue with its f_score, g_score, and updated path
                    heapq.heappush(priority_queue, (f_score_new_state, new_g_score, new_state, path + [new_state]))
    return None # No solution found

# Example usage:
# A solvable initial state
initial_puzzle_state = (1, 2, 3, 0, 4, 6, 7, 5, 8) # A fairly easy one
# An unsolved initial state (harder one: goal state can be reached, but it requires more moves)
# initial_puzzle_state = (8, 6, 7, 2, 5, 4, 3, 0, 1)

print("Initial Puzzle State:")
print_puzzle(initial_puzzle_state)

print("Searching for solution...")
solution_path = a_star_8_puzzle(initial_puzzle_state)

if solution_path:
    print(f"Solution Found in {len(solution_path) - 1} moves:\n")
    for step, state in enumerate(solution_path):
        print(f"Step {step}:")
        print_puzzle(state)
else:
    print("No solution found for the given initial state.")


Initial Puzzle State:
(1, 2, 3)
(0, 4, 6)
(7, 5, 8)

Searching for solution...
Solution Found in 3 moves:

Step 0:
(1, 2, 3)
(0, 4, 6)
(7, 5, 8)

Step 1:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Step 2:
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Step 3:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

